In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (ResNet18 adapted for CIFAR10)
        resnet = models.resnet18(pretrained=False)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)  # Adapt first layer
        resnet.maxpool = nn.Identity()  # Remove initial maxpool
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128))
        
        # Decoder for 32x32 output
        self.decoder = nn.Sequential(
            nn.Linear(512, 512 * 4 * 4),
            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 16x16 -> 32x32
            nn.ReLU(),
            nn.Conv2d(64, 3, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.encoder(x).flatten(1)
        projected = F.normalize(self.projection(features), p=2, dim=1)
        reconstruction = self.decoder(features)
        return reconstruction, projected

In [2]:
# class Autoencoder(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Enhanced ResNet18 encoder for CIFAR10
#         resnet = models.resnet18(pretrained=False)
#         resnet.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
#         resnet.maxpool = nn.Identity()
#         self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
#         # Stronger projection head
#         self.projection = nn.Sequential(
#             nn.Linear(512, 1024),
#             nn.BatchNorm1d(1024),
#             nn.ReLU(),
#             nn.Linear(1024, 256),
#             nn.BatchNorm1d(256),
#             nn.ReLU(),
#             nn.Linear(256, 128)
#         )

#         # More powerful decoder
#         self.decoder = nn.Sequential(
#             nn.Linear(512, 512 * 4 * 4),
#             nn.Unflatten(1, (512, 4, 4)),
#             nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 8x8
#             nn.BatchNorm2d(256),
#             nn.ReLU(),
#             nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 16x16
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 32x32
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.Conv2d(64, 3, 3, 1, 1),
#             nn.Sigmoid()
#         )
#     def forward(self, x):
#         features = self.encoder(x).flatten(1)
#         projected = F.normalize(self.projection(features), p=2, dim=1)
#         reconstruction = self.decoder(features)
#         return reconstruction, projected

In [2]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        
        # Compute similarity matrix
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Create mask for positive pairs (excluding self)
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        self_mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        mask = mask - self_mask
        
        # Subtract max for numerical stability
        max_sim, _ = torch.max(similarity_matrix, dim=1, keepdim=True)
        logits = similarity_matrix - max_sim.detach()
        
        # Compute log probabilities
        exp_logits = torch.exp(logits)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-10)
        
        # Compute mean log prob over positive pairs
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-10)
        mean_log_prob_pos = torch.nan_to_num(mean_log_prob_pos, nan=0.0)
        
        loss = -mean_log_prob_pos.mean()

        return loss

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from torch.utils.data import Dataset
class MemoryDataset(Dataset):
    def __init__(self, dataset):
        self.data = [dataset[i] for i in range(len(dataset))]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# Define data transformations
# train_transform = transforms.Compose([
#     transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),  # Random crop
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))  # CIFAR10 stats
# ])
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, shear=10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Load CIFAR10 dataset
train_val_dataset = datasets.CIFAR10(root='./data', 
                                    train=True, 
                                    download=True,
                                    transform=train_transform)

# Split into train and validation (90% train, 10% val)
train_size = int(0.9 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

# Replace validation set transform (no augmentation)
val_dataset.dataset.transform = val_transform


train_dataset = MemoryDataset(train_dataset) # loads dataset to memory!
val_dataset = MemoryDataset(val_dataset)
# Create dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, 
                         batch_size=batch_size,
                         shuffle=True,
                         num_workers=0,
                         pin_memory=True)

val_loader = DataLoader(val_dataset,
                       batch_size=batch_size,
                       shuffle=False,
                       num_workers=0,
                       pin_memory=True)

# Optional: Test dataset (not used in training)
test_dataset = datasets.CIFAR10(root='./data',
                               train=False,
                               download=True,
                               transform=val_transform)

test_dataset = MemoryDataset(test_dataset)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Print dataset sizes
print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test samples: {len(test_dataset)}')

Train samples: 45000
Validation samples: 5000
Test samples: 10000


In [ ]:
# Initialize model and loss functions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder().to(device)
supcon_criterion = SupConLoss(temperature=0.07)
recon_criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0022, weight_decay=1e-4)
# Training loop
num_epochs = 15
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
contrastive_weight = 0.5  # Adjust based on your needs

In [ ]:
# import torch_lr_finder

# # Run this before training
# lr_finder = torch_lr_finder.LRFinder(model, optimizer, criterion)
# lr_finder.range_test(train_loader, end_lr=1e-2, num_iter=100)
# lr_finder.plot()

In [14]:

print("start epochs")
best_val_loss = float('inf')
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        reconstructions, projections = model(inputs)
        recon_loss = recon_criterion(reconstructions, inputs)
        supcon_loss = supcon_criterion(projections, labels)
        total_loss = (1 - contrastive_weight) * recon_loss + contrastive_weight * supcon_loss
        
        # Backward pass and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        train_loss += total_loss.item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            reconstructions, projections = model(inputs)
            recon_loss = recon_criterion(reconstructions, inputs)
            supcon_loss = supcon_criterion(projections, labels)
            total_loss = recon_loss + contrastive_weight * supcon_loss
            
            val_loss += total_loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    # Save model when validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model_paper.pth')
        print("saved model")
    
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

start epochs
saved model
Epoch [1/7]
Train Loss: 1.1397, Val Loss: 1.1374
saved model
Epoch [2/7]
Train Loss: 1.0251, Val Loss: 1.0663
saved model
Epoch [3/7]
Train Loss: 0.9903, Val Loss: 1.0401


KeyboardInterrupt: 

In [15]:
model.load_state_dict(torch.load('best_model_paper.pth'))
model.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Identity()
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(

In [16]:
# Visualization
import importlib
import utils
importlib.reload(utils)
from utils import plot_tsne
plot_tsne(model.encoder, test_loader, device)

In [8]:
# class Classifier(nn.Module):
#     def __init__(self, pretrained_encoder):
#         super().__init__()
#         # Use pretrained encoder from autoencoder
#         self.encoder = pretrained_encoder
#         # Freeze encoder parameters
#         for param in self.encoder.parameters():
#             param.requires_grad = False
        
#         # Classification head
#         self.fc = nn.Sequential(
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.5),
#             nn.Linear(256, 10)
#         )
class Classifier(nn.Module):
    def __init__(self, pretrained_encoder, input_dim=128, num_classes=10):
        super(Classifier, self).__init__()
        # Freeze encoder parameters
        self.encoder = pretrained_encoder
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        with torch.no_grad():
            features = self.encoder(x).flatten(1)
        return self.head(features)
    # def forward(self, x):
    #     with torch.no_grad():  # No gradient for encoder
    #         features = self.encoder(x).flatten(1)
    #     return self.fc(features)

In [9]:
# Initialize components
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pretrained autoencoder (use your trained model)
autoencoder = Autoencoder().to(device)
# autoencoder.load_state_dict(torch.load('autoencoder.pth'))  # Uncomment if saved

# Create classifier
classifier = Classifier(autoencoder.encoder, input_dim=512).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

In [10]:
classifier_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Reload datasets with new transforms
train_val_dataset = datasets.CIFAR10(root='./data', 
                                   train=True, 
                                   download=True,
                                   transform=classifier_train_transform)  # Changed transform

# Split into train/val (now both using standard transforms)
train_size = int(0.9 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

train_dataset = MemoryDataset(train_dataset) # loads dataset to memory!
val_dataset = MemoryDataset(val_dataset)

# Create new dataloaders
classifier_train_loader = DataLoader(train_dataset, 
                                    batch_size=128,
                                    shuffle=True,
                                    num_workers=0)

classifier_val_loader = DataLoader(val_dataset,
                                  batch_size=128,
                                  shuffle=False,
                                  num_workers=0)



In [11]:
best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    # Training phase
    classifier.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
    
    # Validation phase
    classifier.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = classifier(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
    
    # Calculate metrics
    train_acc = 100. * correct_train / total_train
    val_acc = 100. * correct_val / total_val
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), 'best_classifier.pth')
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}%')
    print(f'Val Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}%')
    print('-' * 50)

Epoch 1/15
Train Loss: 1.9194 | Acc: 29.79%
Val Loss: 1.8298 | Acc: 32.94%
--------------------------------------------------
Epoch 2/15
Train Loss: 1.8102 | Acc: 33.75%
Val Loss: 1.7874 | Acc: 34.80%
--------------------------------------------------
Epoch 3/15
Train Loss: 1.7699 | Acc: 35.67%
Val Loss: 1.7534 | Acc: 38.14%
--------------------------------------------------
Epoch 4/15
Train Loss: 1.7431 | Acc: 36.41%
Val Loss: 1.7607 | Acc: 36.30%
--------------------------------------------------
Epoch 5/15
Train Loss: 1.7178 | Acc: 37.91%
Val Loss: 1.7465 | Acc: 37.08%
--------------------------------------------------
Epoch 6/15
Train Loss: 1.6970 | Acc: 38.32%
Val Loss: 1.7363 | Acc: 37.06%
--------------------------------------------------


KeyboardInterrupt: 

In [25]:
# Load best model
classifier.load_state_dict(torch.load('best_classifier.pth'))
classifier.eval()

test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

avg_test_loss = test_loss / len(test_loader)
test_acc = 100. * correct_test / total_test

print(f'Test Results:')
print(f'Loss: {avg_test_loss:.4f} | Accuracy: {test_acc:.2f}%')

Test Results:
Loss: 0.5773 | Accuracy: 81.69%
